# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records, Demand, Weather, and PLUTO Total Units Dataset:

In this notebook, we will create the final spark dataframe for implenting models. We will firstly merge weather and PLUTO total units data to hourly demand and HVFHV datasets. One-Hot Encoding will also be applied to categorical features.

----

# Import Libraries:

In [1]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql.functions import array_contains, col, explode, lit, when
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import col
from pyspark.sql.functions import * 
from functools import reduce
import pandas as pd 
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_final")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/21 01:43:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Files:

In [3]:
base_dir = "../data"

Hourly demand and HVFHV dataset:

In [4]:
hourly_demand_hvfhv_sdf_dir = base_dir + '/developed/merged_data/hourly_demand_hvfhv'
hourly_demand_hvfhv_sdf = spark.read.parquet(hourly_demand_hvfhv_sdf_dir)
hourly_demand_hvfhv_sdf.show(5)

+-------------+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+------------------+------------------+------------------+------------------------+---------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|         avg_tolls|           avg_bcf|     avg_sales_tax|avg_congestion_surcharge|avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_flag|avg_request_to_pickup_minutes|     avg_trip_speed| avg_total_revenue|day_type|
+-------------+-----------+-----+-----------+------------+--------

Hourly weather dataset:

In [5]:
hourly_weather_sdf_path = base_dir + '/curated/weather_data/preprocessed_hourly_weather'
hourly_weather_sdf = spark.read.parquet(hourly_weather_sdf_path)
hourly_weather_sdf.show(5)

+----------+----+-------------+----------------------+-------------------+------------------+
|  DateOnly|Hour|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHourlyWindSpeed|
+----------+----+-------------+----------------------+-------------------+------------------+
|2023-07-01|   0|         22.2|                   0.0|              9.656|               0.0|
|2023-07-01|   1|         21.7|                   0.0|              9.656|               2.6|
|2023-07-01|   2|         21.1|                   0.0|              9.656|               1.5|
|2023-07-01|   3|         21.1|                   0.0|             11.265|               1.5|
|2023-07-01|   4|         20.6|                   0.0|              9.656|               0.0|
+----------+----+-------------+----------------------+-------------------+------------------+
only showing top 5 rows



PLUTO dataset:

In [6]:
# Read in CSV file
pluto_path = base_dir + '/developed/merged_data/pluto_df.csv'
pluto_df = pd.read_csv(pluto_path)

# Convert Pandas DataFrame to Spark DataFrame
pluto_sdf = spark.createDataFrame(pluto_df)

# Save as Parquet file
pluto_parquet_path = base_dir + '/developed/merged_data/pluto'
pluto_sdf.write.parquet(pluto_parquet_path, mode='overwrite')

# Read back the Parquet file
pluto_sdf = spark.read.parquet(pluto_parquet_path)
pluto_sdf = pluto_sdf.drop('geometry')
pluto_sdf.show(5)

24/08/21 01:44:08 WARN TaskSetManager: Stage 4 contains a task of very large size (5014 KiB). The maximum recommended task size is 1000 KiB.


+--------------+-----------+------------+---------+-------+
|building_class|location_id|service_zone|     zone|borough|
+--------------+-----------+------------+---------+-------+
|             B|         19|   Boro Zone|Bellerose| Queens|
|             B|         19|   Boro Zone|Bellerose| Queens|
|             A|         19|   Boro Zone|Bellerose| Queens|
|             B|         19|   Boro Zone|Bellerose| Queens|
|             A|         19|   Boro Zone|Bellerose| Queens|
+--------------+-----------+------------+---------+-------+
only showing top 5 rows



# Aggregate PLUTO Dataset:

In [7]:
# Remove duplicate rows from DataFrame
pluto_sdf_unique = pluto_sdf.dropDuplicates()

# Aggregate the same location_id, service_zone, zone, 
# borough, and collect unique building_class to a list
pluto_sdf_unique = pluto_sdf_unique.groupBy(
    "location_id", "service_zone", "zone", "borough"
).agg(
    F.collect_set("building_class").alias("building_classes")
)

pluto_sdf_unique.show(5)

+-----------+------------+--------------------+-------------+--------------------+
|location_id|service_zone|                zone|      borough|    building_classes|
+-----------+------------+--------------------+-------------+--------------------+
|          2|   Boro Zone|         Jamaica Bay|       Queens|        [V, G, Q, U]|
|          3|   Boro Zone|Allerton/Pelham G...|        Bronx|[D, K, C, P, Z, S...|
|          4| Yellow Zone|       Alphabet City|    Manhattan|[O, K, C, P, NaN,...|
|          5|   Boro Zone|       Arden Heights|Staten Island|[K, C, Z, S, M, V...|
|          6|   Boro Zone|Arrochar/Fort Wad...|Staten Island|[D, K, C, P, NaN,...|
+-----------+------------+--------------------+-------------+--------------------+
only showing top 5 rows



# Merge Hourly Demand and HVFHV dataset & Hourly Weather Dataset:

In [8]:
# Left join on date and hour
hourly_demand_hvfhv_weather_sdf = hourly_demand_hvfhv_sdf.join(
    hourly_weather_sdf,
    (hourly_demand_hvfhv_sdf.pickup_date == hourly_weather_sdf.DateOnly) &
    (hourly_demand_hvfhv_sdf.pickup_hour == hourly_weather_sdf.Hour),
    how='left'
).drop(hourly_weather_sdf.DateOnly).drop(hourly_weather_sdf.Hour)  # 删除重复的 Hour 列

hourly_demand_hvfhv_weather_sdf.show(5)

+-------------+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+------------------+------------------+------------------+------------------------+---------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+-------------+----------------------+-------------------+------------------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|         avg_tolls|           avg_bcf|     avg_sales_tax|avg_congestion_surcharge|avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_flag|avg_request_to_pickup_minutes|     avg_trip_speed| avg_total_revenue|

In [9]:
num_rows = hourly_demand_hvfhv_weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1108306
Number of columns: 29


# Merge the Above Dataset with PLUTO Dataset:

In [10]:
# Left join on location ID
hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_sdf.join(
    pluto_sdf_unique,
    hourly_demand_hvfhv_weather_sdf.PULocationID == pluto_sdf_unique.location_id,
    how='left'
).drop(pluto_sdf_unique.location_id) 

hourly_demand_hvfhv_weather_pluto_sdf.show(5)

+-------------+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+-------------------+------------------+------------------+------------------------+---------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+-------------+----------------------+-------------------+------------------+------------+--------------------+-------------+--------------------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|          avg_tolls|           avg_bcf|     avg_sales_tax|avg_congestion_surcharge|avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_fla

In [11]:
num_rows = hourly_demand_hvfhv_weather_pluto_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_weather_pluto_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1108306
Number of columns: 33


In [12]:
# Calculate the amount of NULL in each column
null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in hourly_demand_hvfhv_weather_pluto_sdf.columns]
na_counts = hourly_demand_hvfhv_weather_pluto_sdf.agg(*null_counts_expr)
na_counts.show()

+-------------+-----------+-----+-----------+------------+--------------+-------------+--------------------+-----------------------+---------------------+---------+-------+-------------+------------------------+---------------+--------+--------------+-----------------------+---------------------+--------------------+------------------+-----------------------------+--------------+-----------------+--------+-------------+----------------------+-------------------+------------------+------------+----+-------+----------------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|avg_trip_miles|avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|avg_tolls|avg_bcf|avg_sales_tax|avg_congestion_surcharge|avg_airport_fee|avg_tips|avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|avg_wav_match_flag|avg_request_to_pickup_minutes|avg_trip_speed|avg_total_revenue|day_type|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHo

In [13]:
# Filtere out the rows are all NULLs for particular columns
columns_to_check = ['service_zone', 'zone', 'borough', 'building_classes']

null_conditions = [F.col(c).isNull() for c in columns_to_check]
combined_condition = reduce(lambda a, b: a & b, null_conditions)
filtered_df = hourly_demand_hvfhv_weather_pluto_sdf.filter(combined_condition)

filtered_df.show()


+-------------+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+------------------+-------------------+------------------+------------------------+---------------+-------------------+------------------+-----------------------+---------------------+--------------------+-------------------+-----------------------------+-------------------+------------------+--------+------------------+----------------------+-------------------+------------------+------------+----+-------+----------------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|         avg_tolls|            avg_bcf|     avg_sales_tax|avg_congestion_surcharge|avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag| avg_wav_match_flag|avg_request_to_pickup

We found there are lots of NULLs for location 57 in the above dataset, and there is no records for location 57 in PLUTO dataset. So we will remove the rows which are locaiton 57.

In [14]:
pluto_sdf_unique.filter(col('location_id') == 57).show()

+-----------+------------+----+-------+----------------+
|location_id|service_zone|zone|borough|building_classes|
+-----------+------------+----+-------+----------------+
+-----------+------------+----+-------+----------------+



In [15]:
hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_pluto_sdf.filter(col('location_id') != 57)

In [16]:
num_rows = hourly_demand_hvfhv_weather_pluto_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_weather_pluto_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1103947
Number of columns: 33


Confirm there is no NULLs:

In [17]:
# Calculate the amount of NULL in each column
null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in hourly_demand_hvfhv_weather_pluto_sdf.columns]
na_counts = hourly_demand_hvfhv_weather_pluto_sdf.agg(*null_counts_expr)
na_counts.show()

+-------------+-----------+-----+-----------+------------+--------------+-------------+--------------------+-----------------------+---------------------+---------+-------+-------------+------------------------+---------------+--------+--------------+-----------------------+---------------------+--------------------+------------------+-----------------------------+--------------+-----------------+--------+-------------+----------------------+-------------------+------------------+------------+----+-------+----------------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|avg_trip_miles|avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|avg_tolls|avg_bcf|avg_sales_tax|avg_congestion_surcharge|avg_airport_fee|avg_tips|avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|avg_wav_match_flag|avg_request_to_pickup_minutes|avg_trip_speed|avg_total_revenue|day_type|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHo

In [18]:
hourly_demand_hvfhv_weather_pluto_sdf.show(5)

+-------------+-----------+-----+-----------+------------+-----------------+------------------+--------------------+-----------------------+---------------------+-------------------+-------------------+------------------+------------------------+-------------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+-----------------+----------------------+-------------------+------------------+------------+--------------+--------+--------------------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|   avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|          avg_tolls|            avg_bcf|     avg_sales_tax|avg_congestion_surcharge|    avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_fl

# Drop `pickup_date` in the Above Combined Dataset and Aggregated Again:

In [19]:
# Drop the 'pickup_date' column
hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_pluto_sdf.drop("pickup_date")

# List of columns to group by
group_by_columns = [
    "month", "pickup_hour", "avg_utilization_rate", "PULocationID", "avg_trip_miles", "avg_trip_time",
    "avg_base_passenger_fare", "avg_total_fare_amount", "avg_tolls", "avg_bcf",
    "avg_sales_tax", "avg_congestion_surcharge", "avg_airport_fee", "avg_tips",
    "avg_driver_pay", "avg_shared_request_flag", "avg_shared_match_flag",
    "avg_wav_request_flag", "avg_wav_match_flag", "avg_request_to_pickup_minutes",
    "avg_trip_speed", "avg_total_revenue", "day_type", "AvgHourlyTemp",
    "AvgHourlyPrecipitation", "AvgHourlyVisibility", "AvgHourlyWindSpeed",
    "service_zone", "zone", "borough", "building_classes"
]

# Aggregate the data
hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_pluto_sdf.groupBy(group_by_columns).agg(avg("hourly_demand").alias("avg_hourly_demand"))

# Show the result
hourly_demand_hvfhv_weather_pluto_sdf.show(5)

+-----+-----------+--------------------+------------+------------------+------------------+-----------------------+---------------------+--------------------+-------------------+------------------+------------------------+-------------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+-------------+----------------------+-------------------+------------------+------------+--------------+--------+--------------------+-----------------+
|month|pickup_hour|avg_utilization_rate|PULocationID|    avg_trip_miles|     avg_trip_time|avg_base_passenger_fare|avg_total_fare_amount|           avg_tolls|            avg_bcf|     avg_sales_tax|avg_congestion_surcharge|    avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_flag|avg_request_to_pickup_minutes| 

In [20]:
num_rows = hourly_demand_hvfhv_weather_pluto_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_weather_pluto_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1103947
Number of columns: 32


# Save the Merged Dataset Related to Hourly Demand, HVFHV, Weather, and PLUTO Datasets:

In [21]:
merge_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_hvfhv_weather_pluto'
merge_path = os.path.join(merge_dir, file_name)
hourly_demand_hvfhv_weather_pluto_sdf.write.mode('overwrite').parquet(merge_path)

# Apply One-Hot Encoding to Discrete Features:

#### For `PULocationID`, `service_zone`, `zone`, `borough` columns:

In [22]:
# Collect unique labels for each feature column
feature_columns = ["PULocationID", "service_zone", "zone", "borough"]
unique_labels = {col: hourly_demand_hvfhv_weather_pluto_sdf.select(col).distinct().rdd.flatMap(lambda x: x).collect() for col in feature_columns}

# One-Hot Encoding for each feature column
for feature in feature_columns:
    labels = unique_labels[feature]
    for label in labels:
        # Add a new column to indicate whether each row contains the label
        hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_pluto_sdf.withColumn(f"{feature}_{label}", when(col(feature) == label, 1).otherwise(0))

# Drop unused columns
hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_pluto_sdf.drop(
    "PULocationID", 
    "service_zone", 
    "zone", 
    "borough"
)

hourly_demand_hvfhv_weather_pluto_sdf.show(5)

+-----+-----------+--------------------+------------------+------------------+-----------------------+---------------------+--------------------+-------------------+------------------+------------------------+-------------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+-------------+----------------------+-------------------+------------------+--------------------+-----------------+----------------+----------------+---------------+---------------+----------------+----------------+---------------+---------------+----------------+----------------+---------------+----------------+----------------+----------------+----------------+---------------+----------------+----------------+----------------+---------------+---------------+----------------+----------------+---------------+---------------+---------------+---------------+----

#### For `building_classes`:

In [23]:
# Expand the `building_classes` column
exploded_df = hourly_demand_hvfhv_weather_pluto_sdf.withColumn("building_class", explode(col("building_classes")))

# Get all unique tags
distinct_labels = exploded_df.select("building_class").distinct().rdd.flatMap(lambda x: x).collect()

# Add binary columns for each label
for label in distinct_labels:
    hourly_demand_hvfhv_weather_pluto_sdf = hourly_demand_hvfhv_weather_pluto_sdf.withColumn(
        label,
        when(array_contains(col("building_classes"), label), lit(1)).otherwise(lit(0))
    )

hourly_demand_hvfhv_weather_pluto_sdf.drop('building_classes')

hourly_demand_hvfhv_weather_pluto_sdf.show(5)

+-----+-----------+--------------------+------------------+------------------+-----------------------+---------------------+--------------------+-------------------+------------------+------------------------+-------------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+-------------+----------------------+-------------------+------------------+--------------------+-----------------+----------------+----------------+---------------+---------------+----------------+----------------+---------------+---------------+----------------+----------------+---------------+----------------+----------------+----------------+----------------+---------------+----------------+----------------+----------------+---------------+---------------+----------------+----------------+---------------+---------------+---------------+---------------+----

In [24]:
num_rows = hourly_demand_hvfhv_weather_pluto_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_weather_pluto_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1103947
Number of columns: 577


In [25]:
hourly_demand_hvfhv_weather_pluto_sdf.printSchema()

root
 |-- month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- avg_utilization_rate: double (nullable = true)
 |-- avg_trip_miles: double (nullable = true)
 |-- avg_trip_time: double (nullable = true)
 |-- avg_base_passenger_fare: double (nullable = true)
 |-- avg_total_fare_amount: double (nullable = true)
 |-- avg_tolls: double (nullable = true)
 |-- avg_bcf: double (nullable = true)
 |-- avg_sales_tax: double (nullable = true)
 |-- avg_congestion_surcharge: double (nullable = true)
 |-- avg_airport_fee: double (nullable = true)
 |-- avg_tips: double (nullable = true)
 |-- avg_driver_pay: double (nullable = true)
 |-- avg_shared_request_flag: double (nullable = true)
 |-- avg_shared_match_flag: double (nullable = true)
 |-- avg_wav_request_flag: double (nullable = true)
 |-- avg_wav_match_flag: double (nullable = true)
 |-- avg_request_to_pickup_minutes: double (nullable = true)
 |-- avg_trip_speed: double (nullable = true)
 |-- avg_total_revenue: double 

# Save the Final Combined Dataset:

In [26]:
final_sdf_dir = base_dir + '/developed'
file_name = 'final_data'
final_sdf_path = os.path.join(final_sdf_dir, file_name)
hourly_demand_hvfhv_weather_pluto_sdf.write.mode('overwrite').parquet(final_sdf_path)